# 08 - Primeiro trabalho
Em dupla ou trio, faça de acordo com a especificação do trabalho, usando os conceitos de método da bisseção, métodos de ponto fixo, métodos de Gauss e resolução de sistemas não lineares.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana8
```

Faça o checkout nessa nova branch:

```bash
git checkout semana8
```

<hr />

## Atividade 1
Para economizar energia elétrica, um agricultor implantou um sistema de painel solar na fazenda para alimentar uma bomba d’água, que faz a irrigação das plantações.  
A placa negra utilizada fica exposta ao sol e ao vento. Dessa forma, para gerar eletricidade, a irradiação solar $E$ (em $\text{W/m}^2$) sobre a placa tem que ser maior que a perda de calor por radiação e por condução, dada pela temperatura atmosférica $K$ (em Kelvin).

A equação é dada por:

$$
E = 5,67 \times 10^{-8} \, T^4 + 0,4(T - K)
$$

Determine a temperatura mínima da placa, dada a medição da irradiação e da temperatura do ar, sabendo que o valor médio diário é $E = 500,125$ e $K = 272,975$.

**Sugestão:** usar métodos da bissecção ou ponto fixo.  
**Resposta aproximada:**  

$$
T = 304{,}56801011987010952
$$

In [ ]:
E = 500.125
K = 272.975


def residuo(T):
    return 5.67e-8 * T**4 + 0.4 * (T - K) - E


esquerda = K
direita = 400.0
tolerancia = 1e-12

if residuo(esquerda) * residuo(direita) >= 0:
    raise ValueError("O intervalo inicial nao contem uma raiz.")

iteracoes = 0
while (direita - esquerda) / 2 > tolerancia:
    meio = (esquerda + direita) / 2
    if residuo(meio) == 0:
        esquerda = direita = meio
        break
    if residuo(esquerda) * residuo(meio) < 0:
        direita = meio
    else:
        esquerda = meio
    iteracoes += 1

temperatura = (esquerda + direita) / 2
print(f"Temperatura minima da placa: T = {temperatura:.14f} K")
print(f"Iteracoes: {iteracoes}")
print(f"Residuo: {residuo(temperatura):.3e} W/m^2")

## Atividade 2
Em um grande hospital, para evitar intermitência no sistema elétrico dos diversos equipamentos médicos, instalou-se três reatores de alta performance em série, com tensão nominal $R$ em função da potência máxima $P$ de operação dos enrolamentos de controle:

$$
17R_1 - 2R_2 - 3R_3 = P_1 = 500
$$  

$$
-5R_1 + 21R_2 - 2R_3 = P_2 = 200
$$  

$$
-5R_1 - 5R_2 + 22R_3 = P_3 = 300
$$  

Calcular as tensões nominais de cada reator.

**Sugestão:** usar o método de Gauss-Seidel.  
**Resposta aproximada:**  

$$
R_1 = 36{,}56081655798128
$$  

$$
R_2 = 20{,}768358378225123
$$  

$$
R_3 = 26{,}66572157641055
$$  


In [ ]:
import numpy as np

A = np.array([
    [17.0, -2.0, -3.0],
    [-5.0, 21.0, -2.0],
    [-5.0, -5.0, 22.0],
])
b = np.array([500.0, 200.0, 300.0])
tolerancia = 1e-10
R = np.zeros(3)

print(f"{'k':>2} {'R1':>16} {'R2':>16} {'R3':>16} {'erro':>12}")
print(f"{0:2d} {R[0]:16.10f} {R[1]:16.10f} {R[2]:16.10f}")

for k in range(1, 1001):
    anterior = R.copy()
    R[0] = (b[0] + 2*R[1] + 3*R[2]) / 17
    R[1] = (b[1] + 5*R[0] + 2*R[2]) / 21
    R[2] = (b[2] + 5*R[0] + 5*R[1]) / 22
    erro = np.linalg.norm(R - anterior)
    print(f"{k:2d} {R[0]:16.10f} {R[1]:16.10f} {R[2]:16.10f} {erro:12.3e}")
    if erro < tolerancia:
        break
else:
    raise RuntimeError("Gauss-Seidel nao convergiu.")

solucao_direta = np.linalg.solve(A, b)
assert np.allclose(R, solucao_direta, atol=1e-9)
print(f"\\nTensoes nominais: R1={R[0]:.14f}, R2={R[1]:.14f}, R3={R[2]:.14f}")
print("Solucao direta:", solucao_direta)

## Atividade 3
Determine o valor da corrente no resistor **R3**, sabendo que $V_1 = V_2 = 100\,V$ e $R_1 = R_2 = R_3 = 10\,\Omega$.

### Circuito equivalente

            R2           R1
     ┌─────/\/\/\───┬──/\/\/\───┐
     │             │            │
     │             │            │
    V2(+)        /\/\/\      V1(+)
     │            R3            │
     │             │            │
     └─────────────┴────────────┘

$$
\begin{bmatrix}
20 & 10 \\
10 & 20
\end{bmatrix}
\begin{Bmatrix}
I_1 \\
I_2
\end{Bmatrix}
=
\begin{Bmatrix}
100 \\
100
\end{Bmatrix}
$$

**Sugestão:** usar o método de Gauss-Seidel.

**Resposta aproximada:**  

$$
I_{R3} = 6{,}6667\,A
$$

In [ ]:
import numpy as np

A = np.array([[20.0, 10.0], [10.0, 20.0]])
b = np.array([100.0, 100.0])
tolerancia = 1e-10
I1, I2 = 0.0, 0.0

for k in range(1, 1001):
    anterior = np.array([I1, I2])
    I1 = (100.0 - 10.0 * I2) / 20.0
    I2 = (100.0 - 10.0 * I1) / 20.0
    correntes = np.array([I1, I2])
    erro = np.linalg.norm(correntes - anterior)
    if erro < tolerancia:
        break
else:
    raise RuntimeError("Gauss-Seidel nao convergiu.")

solucao_direta = np.linalg.solve(A, b)
corrente_R3 = I1 + I2
assert np.allclose(correntes, solucao_direta, atol=1e-9)
assert np.isclose(corrente_R3, 100.0 / 15.0, atol=1e-9)

print(f"Iteracoes de Gauss-Seidel: {k}")
print(f"I1 = {I1:.10f} A, I2 = {I2:.10f} A")
print(f"Corrente em R3: I_R3 = I1 + I2 = {corrente_R3:.4f} A")
print("Solucao direta:", solucao_direta)

## Atividade 4

Num sistema de energia solar, um vetor de equilíbrio energético $E$, na placa absorvente e na placa de vidro, segue o seguinte sistema de equações não lineares nas temperaturas absolutas $(K/m^2)$:

$$
(T_1^4 + 0,06823T_1) - (T_2^4 + 0,05848T_2) = E_1
$$    

$$
(T_1^4 + 0,05848T_1) - (2T_2^4 + 0,11696T_2) = E_2
$$  

Calcule o valor aproximado das temperaturas de equilíbrio, sabendo que os valores de equilíbrio energético foram medidos experimentalmente em:  

$E_1 = 0,01753$ e $E_2 = 0,00254$.

**Sugestão:** usar o método de Newton-Raphson.  
**Resposta aproximada:**  

$$
T_1 = 0,30543
$$  

$$ 
T_2 = 0,185261
$$ 

In [ ]:
import numpy as np

E1 = 0.01753
E2 = 0.00254


def F(T):
    T1, T2 = T
    return np.array([
        T1**4 + 0.06823*T1 - T2**4 - 0.05848*T2 - E1,
        T1**4 + 0.05848*T1 - 2*T2**4 - 0.11696*T2 - E2,
    ])


def jacobiano(T):
    T1, T2 = T
    return np.array([
        [4*T1**3 + 0.06823, -(4*T2**3 + 0.05848)],
        [4*T1**3 + 0.05848, -(8*T2**3 + 0.11696)],
    ])


T = np.array([0.3, 0.2], dtype=float)
tolerancia = 1e-12
print(f"{'k':>2} {'T1':>14} {'T2':>14} {'||F(T)||':>14}")

for k in range(1, 101):
    passo = np.linalg.solve(jacobiano(T), -F(T))
    T = T + passo
    residuo = np.linalg.norm(F(T))
    print(f"{k:2d} {T[0]:14.10f} {T[1]:14.10f} {residuo:14.3e}")
    if np.linalg.norm(passo) < tolerancia and residuo < tolerancia:
        break
else:
    raise RuntimeError("Newton-Raphson nao convergiu.")

print(f"\\nTemperaturas de equilibrio: T1={T[0]:.10f}, T2={T[1]:.10f}")

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 8"
git push origin semana8
```